				# %% [markdown]
# # Feature Engineering — Reply Sandbox 2026 (Level 1)
#
# ## Goals
# This notebook builds a rich feature matrix from the raw Level 1 dataset for the Reply Sandbox 2026 challenge.
#
# ### Feature groups constructed:
# - **Temporal features**: hour, day of week, is_weekend, is_night, month
# - **Rolling features**: rolling mean & std (window=3) per CitizenID for health indices
# - **Lag features**: lag-1 and lag-2 values per CitizenID for health indices
# - **EventType encoding**: one-hot encoding of categorical event types
# - **Geospatial features**: distance from city centroid, event frequency per citizen
# - **Aggregate features**: per-CitizenID mean, std, min, max of health indices
#
# ### Target
# Binary label: `1` if `EventType == "WNACROYX"`, `0` otherwise (manually assigned).

In [8]:
# %%
# =============================================================================
# Cell 1 — Imports and Configuration
# =============================================================================

import warnings
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ── Fix working directory ─────────────────────────────────────────────────────
# Risale da notebooks/ alla root del progetto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(BASE_DIR)
print(f"✅ Working directory: {BASE_DIR}")

# ── Paths ─────────────────────────────────────────────────────────────────────
LEVEL = "public_lev_1"
STATUS_FILE    = BASE_DIR / f"data/raw/{LEVEL}/status.csv"
LOCATIONS_FILE = BASE_DIR / f"data/raw/{LEVEL}/locations.json"
USERS_FILE     = BASE_DIR / f"data/raw/{LEVEL}/users.json"

# Verifica esistenza file
for f in [STATUS_FILE, LOCATIONS_FILE, USERS_FILE]:
    print(f"{'✅' if f.exists() else '❌'} {f}")

# ── Label config ──────────────────────────────────────────────────────────────
MANUAL_LABELS = {
    "IAFGUHCK": 0,  # Christiane - stabile
    "RFLFWVQA": 0,  # Denise - sana e attiva
    "IXTDRHTR": 0,  # Natasha - condizioni gestite
    "WNACROYX": 1,  # Craig - segnali di rischio
    "DCGGXUWF": 0,  # George - ben supportato
}
TARGET_EVENT_TYPE = "follow-up assessment"
print("✅ Imports and configuration complete.")


✅ Working directory: c:\Progetti\Reply-sandbox-2026-the-eye
✅ c:\Progetti\Reply-sandbox-2026-the-eye\data\raw\public_lev_1\status.csv
✅ c:\Progetti\Reply-sandbox-2026-the-eye\data\raw\public_lev_1\locations.json
✅ c:\Progetti\Reply-sandbox-2026-the-eye\data\raw\public_lev_1\users.json
✅ Imports and configuration complete.


In [10]:
# %%
# =============================================================================
# Cell 2 — Load Datasets and Re-Apply Manual Labels
# =============================================================================

# ── status.csv ────────────────────────────────────────────────────────────────
status_df = pd.read_csv(STATUS_FILE, parse_dates=["Timestamp"])
print(f"status.csv loaded  → shape: {status_df.shape}")
print(status_df.dtypes)
print(status_df.head(3))

# Re-apply manual label (overwrite any pre-existing 'label' column if present)
status_df["label"] = (status_df["EventType"] == TARGET_EVENT_TYPE).astype(int)
print(f"\nLabel distribution:\n{status_df['label'].value_counts()}")

# ── locations.json ────────────────────────────────────────────────────────────
with open(LOCATIONS_FILE, "r") as fh:
    locations_raw = json.load(fh)

if isinstance(locations_raw, list):
    locations_df = pd.DataFrame(locations_raw)
else:
    locations_df = pd.json_normalize(locations_raw)

# Standardise BioTag → CitizenID
if "BioTag" in locations_df.columns:
    locations_df.rename(columns={"BioTag": "CitizenID"}, inplace=True)

# ── Detect datetime column dinamicamente ──────────────────────────────────────
print(f"Locations columns: {locations_df.columns.tolist()}")

DT_CANDIDATES = ["datetime", "Datetime", "timestamp", "Timestamp", "date", "Date", "time"]
dt_col = next((c for c in DT_CANDIDATES if c in locations_df.columns), None)

if dt_col:
    locations_df[dt_col] = pd.to_datetime(locations_df[dt_col], errors="coerce")
    locations_df.rename(columns={dt_col: "datetime"}, inplace=True)
    print(f"✅ Datetime column found: '{dt_col}'")
else:
    print(f"⚠️ No datetime column found — skipping datetime parsing")
    locations_df["datetime"] = pd.NaT

locations_df["lat"] = pd.to_numeric(locations_df["lat"], errors="coerce")
locations_df["lng"] = pd.to_numeric(locations_df["lng"], errors="coerce")

print(f"\nlocations.json loaded → shape: {locations_df.shape}")
print(locations_df.head(3))

# ── users.json ────────────────────────────────────────────────────────────────
with open(USERS_FILE, "r") as fh:
    users_raw = json.load(fh)

if isinstance(users_raw, list):
    users_df = pd.DataFrame(users_raw)
else:
    users_df = pd.json_normalize(users_raw)

# Ensure a CitizenID column exists (common alternative names)
for alias in ("id", "ID", "citizen_id", "BioTag", "userId"):
    if alias in users_df.columns and "CitizenID" not in users_df.columns:
        users_df.rename(columns={alias: "CitizenID"}, inplace=True)

print(f"\nusers.json loaded     → shape: {users_df.shape}")
print(users_df.head(3))

status.csv loaded  → shape: (50, 7)
EventID                                int64
CitizenID                                str
EventType                                str
PhysicalActivityIndex                  int64
SleepQualityIndex                      int64
EnvironmentalExposureLevel             int64
Timestamp                     datetime64[us]
dtype: object
   EventID CitizenID                   EventType  PhysicalActivityIndex  \
0        1  IAFGUHCK            routine check-up                     31   
1        2  IAFGUHCK        preventive screening                     37   
2        3  IAFGUHCK  lifestyle coaching session                     34   

   SleepQualityIndex  EnvironmentalExposureLevel           Timestamp  
0                 42                          33 2026-01-05 13:41:36  
1                 52                          30 2026-01-29 06:03:18  
2                 49                          31 2026-03-04 15:46:11  

Label distribution:
label
0    47
1     3
Name: c

In [11]:
# %%
# =============================================================================
# Cell 3 — Temporal Features
# =============================================================================

df = status_df.copy()

# Ensure Timestamp is datetime (belt-and-suspenders)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

df["hour"]        = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek          # 0=Monday … 6=Sunday
df["month"]       = df["Timestamp"].dt.month
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)  # Sat=5, Sun=6

# is_night: True when hour is in [22, 23] OR [0, 1, 2, 3, 4, 5]
df["is_night"] = (
    (df["hour"] >= NIGHT_START) | (df["hour"] < NIGHT_END)
).astype(int)

TEMPORAL_COLS = ["hour", "day_of_week", "month", "is_weekend", "is_night"]
print(f"✅ Temporal features added: {TEMPORAL_COLS}")
print(df[TEMPORAL_COLS].describe())


✅ Temporal features added: ['hour', 'day_of_week', 'month', 'is_weekend', 'is_night']
            hour  day_of_week      month  is_weekend   is_night
count  50.000000    50.000000  50.000000   50.000000  50.000000
mean   12.360000     2.820000   6.240000    0.240000   0.420000
std     7.761128     2.076988   3.390021    0.431419   0.498569
min     0.000000     0.000000   1.000000    0.000000   0.000000
25%     6.250000     1.000000   3.250000    0.000000   0.000000
50%    13.000000     3.000000   6.500000    0.000000   0.000000
75%    18.000000     4.000000   9.000000    0.000000   1.000000
max    23.000000     6.000000  12.000000    1.000000   1.000000


In [12]:
# %%
# =============================================================================
# Cell 4 — Rolling Features per CitizenID
# =============================================================================
# Sort chronologically so rolling windows respect event order.
df.sort_values(["CitizenID", "Timestamp"], inplace=True)
df.reset_index(drop=True, inplace=True)

rolling_cols_added = []

for col in HEALTH_COLS:
    mean_col = f"{col}_roll{ROLLING_WINDOW}_mean"
    std_col  = f"{col}_roll{ROLLING_WINDOW}_std"

    rolled = (
        df.groupby("CitizenID")[col]
          .transform(lambda x: x.rolling(window=ROLLING_WINDOW, min_periods=1).mean())
    )
    df[mean_col] = rolled

    rolled_std = (
        df.groupby("CitizenID")[col]
          .transform(lambda x: x.rolling(window=ROLLING_WINDOW, min_periods=1).std())
    )
    # Fill NaN std (single observation) with 0
    df[std_col] = rolled_std.fillna(0)

    rolling_cols_added.extend([mean_col, std_col])

print(f"✅ Rolling features added ({len(rolling_cols_added)}): {rolling_cols_added}")

✅ Rolling features added (6): ['PhysicalActivityIndex_roll3_mean', 'PhysicalActivityIndex_roll3_std', 'SleepQualityIndex_roll3_mean', 'SleepQualityIndex_roll3_std', 'EnvironmentalExposureLevel_roll3_mean', 'EnvironmentalExposureLevel_roll3_std']


In [13]:
# %%
# =============================================================================
# Cell 5 — Lag Features per CitizenID
# =============================================================================

lag_cols_added = []

for col in HEALTH_COLS:
    for lag in LAG_STEPS:
        lag_col = f"{col}_lag{lag}"
        df[lag_col] = (
            df.groupby("CitizenID")[col]
              .transform(lambda x, l=lag: x.shift(l))
        )
        # Fill initial NaN lags with the column's global median (safe imputation)
        df[lag_col].fillna(df[col].median(), inplace=True)
        lag_cols_added.append(lag_col)

print(f"✅ Lag features added ({len(lag_cols_added)}): {lag_cols_added}")

✅ Lag features added (6): ['PhysicalActivityIndex_lag1', 'PhysicalActivityIndex_lag2', 'SleepQualityIndex_lag1', 'SleepQualityIndex_lag2', 'EnvironmentalExposureLevel_lag1', 'EnvironmentalExposureLevel_lag2']


In [14]:
# %%
# =============================================================================
# Cell 6 — EventType One-Hot Encoding
# =============================================================================

event_dummies = pd.get_dummies(df["EventType"], prefix="event", dtype=int)
df = pd.concat([df, event_dummies], axis=1)

event_ohe_cols = list(event_dummies.columns)
print(f"✅ EventType one-hot encoded → {len(event_ohe_cols)} columns:")
print(event_ohe_cols)

✅ EventType one-hot encoded → 5 columns:
['event_follow-up assessment', 'event_lifestyle coaching session', 'event_preventive screening', 'event_routine check-up', 'event_specialist consultation']


In [25]:
# =============================================================================
# Cell 7 — Geospatial Features from locations.json
# =============================================================================

# ── Pulizia colonne duplicate da riesecuzioni precedenti ─────────────────────
cols_to_drop = [c for c in locations_df.columns 
                if any(x in c for x in ["centroid", "dist_from_centroid"])]
locations_df.drop(columns=cols_to_drop, inplace=True)
print(f"Columns after cleanup: {locations_df.columns.tolist()}")

# ── Rinomina user_id → CitizenID ──────────────────────────────────────────────
if "user_id" in locations_df.columns and "CitizenID" not in locations_df.columns:
    locations_df.rename(columns={"user_id": "CitizenID"}, inplace=True)
    print("✅ Renamed 'user_id' → 'CitizenID'")

# ── 7a. Compute per-city centroid ─────────────────────────────────────────────
city_centroids = (
    locations_df.dropna(subset=["city", "lat", "lng"])
    .groupby("city")[["lat", "lng"]]
    .mean()
    .rename(columns={"lat": "city_lat_centroid", "lng": "city_lng_centroid"})
    .reset_index()
)

# ── 7b. Merge centroid ────────────────────────────────────────────────────────
locations_df = locations_df.merge(city_centroids, on="city", how="left")
print(f"✅ Centroids merged → shape: {locations_df.shape}")

# ── 7c. Haversine distance ────────────────────────────────────────────────────
def haversine_np(lat1, lng1, lat2, lng2):
    R = 6371.0
    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

locations_df["dist_from_centroid_km"] = haversine_np(
    locations_df["lat"].values,
    locations_df["lng"].values,
    locations_df["city_lat_centroid"].values,
    locations_df["city_lng_centroid"].values,
)

# ── 7d. Aggregate geo features per CitizenID ──────────────────────────────────
geo_agg = (
    locations_df.groupby("CitizenID")
    .agg(
        geo_event_count        = ("datetime", "count"),
        geo_mean_lat           = ("lat", "mean"),
        geo_mean_lng           = ("lng", "mean"),
        geo_std_lat            = ("lat", "std"),
        geo_std_lng            = ("lng", "std"),
        geo_mean_dist_centroid = ("dist_from_centroid_km", "mean"),
        geo_max_dist_centroid  = ("dist_from_centroid_km", "max"),
        geo_unique_cities      = ("city", "nunique"),
    )
    .reset_index()
    .fillna(0)
)
print(f"✅ Geo aggregation shape: {geo_agg.shape}")
print(geo_agg)

Columns after cleanup: ['user_id', 'datetime', 'lat', 'lng', 'city']
✅ Renamed 'user_id' → 'CitizenID'
✅ Centroids merged → shape: (958, 7)
✅ Geo aggregation shape: (5, 9)
  CitizenID  geo_event_count  geo_mean_lat  geo_mean_lng  geo_std_lat  \
0  DCGGXUWF              202     41.606091    -79.442122     5.477314   
1  IAFGUHCK              190     48.439255      7.487170     0.635599   
2  IXTDRHTR              192     50.271838      1.677015     5.949704   
3  RFLFWVQA              179     40.684259    -88.479453    17.043387   
4  WNACROYX              195     47.390591     -2.745061    14.861960   

   geo_std_lng  geo_mean_dist_centroid  geo_max_dist_centroid  \
0    25.807913                3.800260               6.725729   
1     1.388113                3.538050               6.798049   
2    25.220664                3.662789               6.805766   
3    24.508459                3.804165               6.973804   
4    16.522140                3.752970               7.367089   

In [26]:
# %%
# =============================================================================
# # Cell 8 — Aggregate Health Features per CitizenID
# =============================================================================
agg_funcs = {col: ["mean", "std", "min", "max"] for col in HEALTH_COLS}

health_agg = (
    df.groupby("CitizenID")
      .agg(agg_funcs)      .reset_index()
)
# Flatten multi-level column names: e.g. ("PhysicalActivityIndex", "mean") → "PhysicalActivityIndex_agg_mean"
health_agg.columns = [    f"{col[0]}_agg_{col[1]}" if col[1] else col[0]
    for col in health_agg.columns]
# Fill NaN std values for citizens with a single event
health_agg.fillna(0, inplace=True)
print(f"✅ Health aggregate features shape: {health_agg.shape}")
print(health_agg.head(3))

✅ Health aggregate features shape: (5, 13)
  CitizenID  PhysicalActivityIndex_agg_mean  PhysicalActivityIndex_agg_std  \
0  DCGGXUWF                            35.9                       3.510302   
1  IAFGUHCK                            34.0                       4.371626   
2  IXTDRHTR                            35.5                       4.169999   

   PhysicalActivityIndex_agg_min  PhysicalActivityIndex_agg_max  \
0                             29                             41   
1                             28                             40   
2                             27                             41   

   SleepQualityIndex_agg_mean  SleepQualityIndex_agg_std  \
0                        47.9                   4.976612   
1                        49.6                   5.059644   
2                        50.1                   3.510302   

   SleepQualityIndex_agg_min  SleepQualityIndex_agg_max  \
0                         39                         56   
1               

In [28]:
# %%
# =============================================================================
# # Cell 9 — Merge All Feature Groups into Final DataFrame
# =============================================================================
# Start from the event-level dataframe (already contains temporal, rolling,
# lag, and OHE features from the previous cells).
final_df = df.copy()
# ── Merge geospatial citizen-level aggregates ──────────────────────────────────
final_df = final_df.merge(geo_agg, on="CitizenID", how="left")

# ── Merge health citizen-level aggregates ─────────────────────────────────────
final_df = final_df.merge(health_agg, on="CitizenID", how="left")
# ── (Optional) Merge user attributes from users.json ─────────────────────────
if "CitizenID" in users_df.columns:
    numeric_user_cols = users_df.select_dtypes(include=[np.number]).columns.tolist()
    user_merge_cols = ["CitizenID"] + [c for c in numeric_user_cols
                                       if c not in final_df.columns]
    final_df = final_df.merge(users_df[user_merge_cols], on="CitizenID", how="left")
    print(f"✅ Users merged → shape: {final_df.shape}")
else:
    print("⚠️ No CitizenID in users_df — skipping user merge")
# ── Drop columns not useful for modelling ─────────────────────────────────────
COLS_TO_DROP = ["EventID", "Timestamp", "EventType"]
final_df.drop(columns=[c for c in COLS_TO_DROP if c in final_df.columns],
              inplace=True)
# ── Ensure no remaining NaNs (fill with column median) ────────────────────────
numeric_cols = final_df.select_dtypes(include=[np.number]).columns
final_df[numeric_cols] = final_df[numeric_cols].fillna(
    final_df[numeric_cols].median()
)
print(f"✅ Final DataFrame shape: {final_df.shape}")
print(final_df.head(3))


⚠️ No CitizenID in users_df — skipping user merge
✅ Final DataFrame shape: (50, 47)
  CitizenID  PhysicalActivityIndex  SleepQualityIndex  \
0  DCGGXUWF                     41                 45   
1  DCGGXUWF                     36                 48   
2  DCGGXUWF                     41                 56   

   EnvironmentalExposureLevel  label  hour  day_of_week  month  is_weekend  \
0                          32      0     0            3      1           0   
1                          27      0    19            0      2           0   
2                          25      0     9            0      3           0   

   is_night  PhysicalActivityIndex_roll3_mean  \
0         1                         41.000000   
1         0                         38.500000   
2         0                         39.333333   

   PhysicalActivityIndex_roll3_std  SleepQualityIndex_roll3_mean  \
0                         0.000000                     45.000000   
1                         3.535534       

In [29]:

# %%
# =============================================================================
# # Cell 10 — Save Feature Matrix to Disk
# =============================================================================
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Feature matrix saved to: {OUTPUT_FILE}")
print(f"   Rows: {final_df.shape[0]:,}  |  Columns: {final_df.shape[1]:,}")


✅ Feature matrix saved to: data\processed\features_lev1.csv
   Rows: 50  |  Columns: 47


In [ ]:
# %%
# =============================================================================
# # Cell 11 — Final Summary: Shape and Feature List
# =============================================================================
feature_cols = [c for c in final_df.columns if c != "label"]

print("=" * 60)
print(f"  Final feature matrix shape : {final_df.shape}")
print(f"  Number of features         : {len(feature_cols)}")
print(f"  Target column              : label")
print(f"  Positive-class rate        : {final_df['label'].mean():.4f}")
print("=" * 60)
print("\n📋 Feature list:")
for i, col in enumerate(feature_cols, 1):
        print(f"  {i:>3}. {col}")

  Final feature matrix shape : (50, 47)
  Number of features         : 46
  Target column              : label
  Positive-class rate        : 0.0600

📋 Feature list:
    1. CitizenID
    2. PhysicalActivityIndex
    3. SleepQualityIndex
    4. EnvironmentalExposureLevel
    5. hour
    6. day_of_week
    7. month
    8. is_weekend
    9. is_night
   10. PhysicalActivityIndex_roll3_mean
   11. PhysicalActivityIndex_roll3_std
   12. SleepQualityIndex_roll3_mean
   13. SleepQualityIndex_roll3_std
   14. EnvironmentalExposureLevel_roll3_mean
   15. EnvironmentalExposureLevel_roll3_std
   16. PhysicalActivityIndex_lag1
   17. PhysicalActivityIndex_lag2
   18. SleepQualityIndex_lag1
   19. SleepQualityIndex_lag2
   20. EnvironmentalExposureLevel_lag1
   21. EnvironmentalExposureLevel_lag2
   22. event_follow-up assessment
   23. event_lifestyle coaching session
   24. event_preventive screening
   25. event_routine check-up
   26. event_specialist consultation
   27. geo_event_count
   28. g